CONFIG (variable)

In [ ]:
try:
    DATASET_NAME = getArgument("DATASET_NAME")
except:
    DATASET_NAME = "Global_Climate_Records"  # ← change to this

StatementMeta(, 563934cc-d2c7-40d0-9123-c54e507cdd8a, 3, Finished, Available, Finished, False)

Ingest Excel into Bronze table (remains universal)

In [ ]:
import pandas as pd

file_path = f"/lakehouse/default/Files/{DATASET_NAME}.xlsx"

df = pd.read_excel(file_path)
df.columns = df.columns.str.lower().str.replace(" ", "_")

TARGET_TABLE = f"bronze_{DATASET_NAME.lower().replace(' ', '_')}"

spark.createDataFrame(df).write.mode("overwrite").saveAsTable(TARGET_TABLE)

print(f"✅ Bronze table created: [{TARGET_TABLE}]")
print(f"📋 Columns: {df.columns.tolist()}")
print(f"📊 Rows: {len(df)}")

StatementMeta(, 563934cc-d2c7-40d0-9123-c54e507cdd8a, 4, Finished, Available, Finished, False)

✅ Bronze table created: [bronze_global_climate_records]
📋 Columns: ['station_id', 'country', 'region', 'observation_date', 'avg_temperature_c', 'rainfall_mm', 'co2_ppm', 'sea_level_mm', 'wind_speed_kmh', 'sensor_status', 'city']
📊 Rows: 307


Define validation rules (RULES are variables)

In [ ]:
# Auto-generate validation rules from any dataset's columns
import pandas as pd

bronze_df = spark.table(TARGET_TABLE).toPandas()
bronze_df.columns = bronze_df.columns.str.lower().str.replace(" ", "_")

auto_rules = []
rule_num = 1

for col, dtype in zip(bronze_df.columns, bronze_df.dtypes):
    auto_rules.append({
        "rule_id":         f"R{rule_num:03d}",
        "column_name":     col,
        "validation_type": "null_check",
        "threshold":       "0"
    })
    rule_num += 1
    if dtype in ["int64", "float64"]:
        auto_rules.append({
            "rule_id":         f"R{rule_num:03d}",
            "column_name":     col,
            "validation_type": "negative_check",
            "threshold":       "0"
        })
        rule_num += 1
    if col == bronze_df.columns[0]:
        auto_rules.append({
            "rule_id":         f"R{rule_num:03d}",
            "column_name":     col,
            "validation_type": "duplicate_check",
            "threshold":       "0.1"
        })
        rule_num += 1
    if "date" in col or "time" in col:
        auto_rules.append({
            "rule_id":         f"R{rule_num:03d}",
            "column_name":     col,
            "validation_type": "freshness_check",
            "threshold":       "720 hrs"
        })
        rule_num += 1

# ← KEY FIX: use dataset-specific rules table name
RULES_TABLE = f"validation_rules_{DATASET_NAME.lower().replace(' ', '_')}"
spark.createDataFrame(pd.DataFrame(auto_rules)) \
     .write.mode("overwrite").saveAsTable(RULES_TABLE)

print(f"✅ {len(auto_rules)} rules generated for [{TARGET_TABLE}]")
display(spark.table(RULES_TABLE))

StatementMeta(, 563934cc-d2c7-40d0-9123-c54e507cdd8a, 5, Finished, Available, Finished, False)

✅ 18 rules generated for [bronze_global_climate_records]


SynapseWidget(Synapse.DataFrame, d3db78f9-4f75-4d30-a121-d4add7acf1a7)

Load data and rules

In [ ]:
rules_df = spark.table(RULES_TABLE).toPandas()
df       = spark.table(TARGET_TABLE).toPandas()
df.columns = df.columns.str.lower().str.replace(" ", "_")

print(f"✅ Loaded {len(df)} rows from [{TARGET_TABLE}]")
print(f"✅ Loaded {len(rules_df)} validation rules")

StatementMeta(, 563934cc-d2c7-40d0-9123-c54e507cdd8a, 6, Finished, Available, Finished, False)

✅ Loaded 307 rows from [bronze_global_climate_records]
✅ Loaded 18 validation rules


 Validation engine (remains universal)

In [ ]:
import ast
from datetime import datetime

results = []

for _, rule in rules_df.iterrows():
    col    = rule["column_name"].lower()
    vtype  = rule["validation_type"]
    thresh = rule["threshold"]

    if col not in df.columns:
        results.append({**rule, "status": "SKIPPED", "detail": "Column not found"})
        continue

    if vtype == "null_check":
        null_pct = df[col].isnull().mean() * 100
        status   = "FAIL" if null_pct > float(thresh) else "PASS"
        results.append({**rule, "status": status, "detail": f"{null_pct:.1f}% nulls"})

    elif vtype == "duplicate_check":
        dup_pct = df[col].duplicated().mean() * 100
        status  = "FAIL" if dup_pct > float(thresh) else "PASS"
        results.append({**rule, "status": status, "detail": f"{dup_pct:.1f}% duplicates"})

    elif vtype == "negative_check":
        neg_count = (df[col] < 0).sum()
        status    = "FAIL" if neg_count > 0 else "PASS"
        results.append({**rule, "status": status, "detail": f"{neg_count} negative values"})

    elif vtype == "accepted_values":
        allowed   = ast.literal_eval(str(thresh))
        allowed   = [v.lower() for v in allowed]
        bad_count = (~df[col].isin(allowed)).sum()
        status    = "FAIL" if bad_count > 0 else "PASS"
        results.append({**rule, "status": status, "detail": f"{bad_count} out-of-range values"})

    elif vtype == "freshness_check":
        max_age_hrs = float(str(thresh).replace(" hrs", "").strip())
        latest      = pd.to_datetime(df[col], errors="coerce").max()
        age_hrs     = (datetime.now() - latest).total_seconds() / 3600 if pd.notna(latest) else 9999
        status      = "FAIL" if age_hrs > max_age_hrs else "PASS"
        results.append({**rule, "status": status, "detail": f"Latest record {age_hrs:.0f}hrs ago"})

validation_df = pd.DataFrame(results)
display(validation_df)

StatementMeta(, 563934cc-d2c7-40d0-9123-c54e507cdd8a, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ccd426b9-d6c9-436c-a692-fef63c7a9f9a)

AI severity scoring (remains universal)

In [ ]:
fail_count  = (validation_df["status"] == "FAIL").sum()
total_rules = len(validation_df)
fail_pct    = fail_count / total_rules * 100

def score_severity(fail_pct):
    if fail_pct >= 50:   return "CRITICAL"
    elif fail_pct >= 25: return "HIGH"
    elif fail_pct > 0:   return "MEDIUM"
    else:                return "HEALTHY"

severity = score_severity(fail_pct)
print(f"🔍 Observability Score for [{DATASET_NAME}]: {severity} ({fail_count}/{total_rules} rules failed)")

StatementMeta(, 563934cc-d2c7-40d0-9123-c54e507cdd8a, 8, Finished, Available, Finished, False)

🔍 Observability Score for [Global_Climate_Records]: CRITICAL (11/18 rules failed)


Write to gold log (remains universal)

In [ ]:
validation_df["dataset_name"] = DATASET_NAME
validation_df["target_table"] = TARGET_TABLE
validation_df["severity"] = severity
validation_df["evaluated_at"] = datetime.now().isoformat()

spark.createDataFrame(validation_df) \
     .write.mode("append") \
     .saveAsTable("Sales_Intelligence_LH.dbo.gold_observability_log")

print(f"✅ Gold log updated for [{TARGET_TABLE}]")

StatementMeta(, 563934cc-d2c7-40d0-9123-c54e507cdd8a, 9, Finished, Available, Finished, False)

✅ Gold log updated for [bronze_global_climate_records]


View gold log (remains universal)

In [ ]:
display(spark.table("gold_observability_log")
    .orderBy("evaluated_at", ascending=False)
    .limit(20))

StatementMeta(, 563934cc-d2c7-40d0-9123-c54e507cdd8a, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1a8f1b36-051f-4804-a0f8-c2b7e7fc4e14)

In [ ]:
display(spark.sql("""
SHOW TABLES IN dbo
"""))

StatementMeta(, bfe9e684-32b8-4c20-a6dd-6de2a9a9b456, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, aeb82ab4-3083-4a83-b02a-c72e1aceeb2c)